# Tool · Notebook 2, "dynamic" version

**What this notebook does:** Like Notebook 2, it turns positions into speeds, with three differences:

* Instead of a fixed 10 ms clock, it makes `UPSAMPLE_FACTOR` times as many measurements as there are video frames
  (so the step size depends on the video's frame rate).
* It smooths more strongly (a longer window).
* It keeps the speed in its raw units (screen widths per millisecond): no scaling to 0-1 and no jitter removal.

| You need | You get |
|---|---|
| `Keypoints/Test_Video_keypoints.csv` (from Notebook 1) | `Upsampled/Test_Video_processed_data_dynamic.csv`: positions and speeds of the chosen body points. |

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import savgol_filter

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"
KEYPOINTS = ["right_wrist", "left_wrist"]   # add or remove body points as needed
UPSAMPLE_FACTOR = 10                        # make 10 times as many measurements as video frames
SMOOTHING_WINDOW = 51                       # must be an odd number

## Step 3 · Load, upsample, smooth and calculate speed

In [ ]:
keypoints = pd.read_csv(f"../Keypoints/{VIDEO_NAME}_keypoints.csv")
start, end = keypoints["time_ms"].iloc[0], keypoints["time_ms"].iloc[-1]
data = pd.DataFrame({"time_ms": np.linspace(start, end, len(keypoints) * UPSAMPLE_FACTOR)})
step_ms = data["time_ms"].iloc[1] - data["time_ms"].iloc[0]

for point in KEYPOINTS:
    for axis in ["x", "y"]:
        column = f"{point}_{axis}"
        tracked = keypoints[column].notna()
        upsampled = np.interp(data["time_ms"], keypoints.loc[tracked, "time_ms"],
                              keypoints.loc[tracked, column])
        data[column] = savgol_filter(upsampled, SMOOTHING_WINDOW, polyorder=3)
    velocity_x = np.gradient(data[f"{point}_x"], step_ms)
    velocity_y = np.gradient(data[f"{point}_y"], step_ms)
    data[f"{point}_speed"] = np.sqrt(velocity_x**2 + velocity_y**2)

print(f"One measurement every {step_ms:.2f} ms")
data.head()

## Step 4 · Save and plot

In [ ]:
Path("../Upsampled").mkdir(exist_ok=True)
output_path = f"../Upsampled/{VIDEO_NAME}_processed_data_dynamic.csv"
data.to_csv(output_path, index=False)
print("Saved:", output_path)

fig = go.Figure()
for point in KEYPOINTS:
    fig.add_scatter(x=data["time_ms"], y=data[f"{point}_speed"], name=point)
fig.update_layout(title="Speed (raw units)", xaxis_title="Time (ms)", yaxis_title="Speed",
                  xaxis_rangeslider_visible=True)
fig.show()